<a href="https://colab.research.google.com/github/tubasa0000/Image2psd/blob/main/layerdivider_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Please set runtime to GPU

In [1]:
!git clone https://github.com/mattyamonaca/layerdivider.git

Cloning into 'layerdivider'...
remote: Enumerating objects: 439, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 439 (delta 127), reused 116 (delta 96), pack-reused 277 (from 1)
Receiving objects: 100% (439/439), 28.47 MiB | 20.72 MiB/s, done.
Resolving deltas: 100% (255/255), done.


In [2]:
%cd layerdivider

/content/layerdivider


In [6]:
# requirements.txt から pandas と opencv のバージョン固定を解除する
!sed -i 's/pandas==1.5.3/pandas/' requirements.txt
!sed -i 's/opencv-python==4.7.0.68/opencv-python/' requirements.txt
!sed -i 's/==[0-9.]*//g' requirements.txt

!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 37.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 845.7/845.7 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.6/255.6 MB 5.6 MB/s eta 0:00:00
  Created wheel for pytoshop: filename=pytoshop-1.2.1-cp313-cp313-linux_x86_64.whl size=600585 sha256=4ca38c7d33c24b667cae0599bbeb43ccc42ceb66f619c33ff905468e3f81526f
  Stored in directory: /root/.cache/pip/wheels/b7/9f/82/d48c8e2d7ce3d7b45e47a2def3e0938be9591ce74415e78e88
Successfully built pytoshop


In [15]:
import os
from huggingface_hub import hf_hub_download

# 1. 必要フォルダの作成
os.makedirs("/content/layerdivider/models", exist_ok=True)
os.makedirs("/content/layerdivider/output/tmp/seg_layer", exist_ok=True)

print("⏳ Meta公式リポジトリから本物のAIモデル(約5GB)を安全にダウンロード中...（2〜3分かかります）")

try:
    # 2. Meta公式のSegment Anythingリポジトリから直ダウンロード（誰でもアクセス可能）
    model_path = hf_hub_download(
        repo_id="facebook/sam-vit-huge",
        filename="pytorch_model.bin"
    )

    # 3. ダウンロードしたファイルをlayerdividerが認識できるファイル名で配置
    target_path = "/content/layerdivider/models/sam_vit_h_4b8939.pth"
    if os.path.exists(target_path):
        os.remove(target_path)

    os.link(model_path, target_path)

    print("✅ ダウンロードと配置が正常に完了しました！")

    # 4. ファイルサイズの確認
    size_bytes = os.path.getsize(target_path)
    print(f"📦 確定ファイルサイズ: {size_bytes / (1024**3):.2f} GB (5.11 GBであれば成功です)")

except Exception as e:
    print(f"❌ エラーが発生しました: {e}")


⏳ Meta公式リポジトリから本物のAIモデル(約5GB)を安全にダウンロード中...（2〜3分かかります）


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.56GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

✅ ダウンロードと配置が正常に完了しました！
❌ エラーが発生しました: [Errno 2] No such file or directory: '/content/layerdivider/models/sam_vit_h_4b8939.pth'


In [21]:
import os
from huggingface_hub import hf_hub_download

print("⏳ 処理を開始します...")

# 1. キャッシュされた本物ファイルの場所を特定（確実に存在します）
model_path = hf_hub_download(repo_id="facebook/sam-vit-huge", filename="pytorch_model.bin")
print(f"📦 本物のモデルファイルを発見しました: {model_path}")

# 2. layerdivider の読み込みコードを書き換える（modelsフォルダを探さずに、上記の本物ファイルを直接読ませる）
ld_utils_path = "/content/layerdivider/ldivider/ld_utils.py"

if os.path.exists(ld_utils_path):
    with open(ld_utils_path, "r") as f:
        code = f.read()

    # 元の「models/sam_vit_h_4b8939.pth」を探すコードを、発見した本物ファイルのフルパスに強制置換
    bug_fixed_code = code.replace('os.path.join("models", "sam_vit_h_4b8939.pth")', f'"{model_path}"')
    bug_fixed_code = bug_fixed_code.replace('"models/sam_vit_h_4b8939.pth"', f'"{model_path}"')

    with open(ld_utils_path, "w") as f:
        f.write(bug_fixed_code)
    print("🛠️ layerdividerのシステムコードを修正し、モデルへの直ルートを開通しました！")

    # 3. 起動する
    print("🚀 layerdivider を起動します...\n" + "="*40)
    os.chdir("/content/layerdivider")
    os.system("python demo.py share")
else:
    print("❌ layerdivider のフォルダが見つかりません。最初のセルからやり直す必要があります。")


⏳ 処理を開始します...
📦 本物のモデルファイルを発見しました: /root/.cache/huggingface/hub/models--facebook--sam-vit-huge/snapshots/87aecf0df4ce6b30cd7de76e87673c49644bdf67/pytorch_model.bin
🛠️ layerdividerのシステムコードを修正し、モデルへの直ルートを開通しました！
🚀 layerdivider を起動します...


In [24]:
import os

print("⚙️ 公開URLの発行を強制的に有効化しています...")

# demo.py のファイルを読み込む
demo_path = "/content/layerdivider/demo.py"

if os.path.exists(demo_path):
    with open(demo_path, "r") as f:
        code = f.read()

    # demo.launch() に share=True を強制的に追加する書き換え
    bug_fixed_code = code.replace("demo.launch()", "demo.launch(share=True)")
    bug_fixed_code = bug_fixed_code.replace("demo.launch(server_name='0.0.0.0')", "demo.launch(server_name='0.0.0.0', share=True)")

    with open(demo_path, "w") as f:
        f.write(bug_fixed_code)
    print("🛠️ 修正が完了しました！ツールを起動します...")

    # そのまま起動
    os.chdir("/content/layerdivider")
    os.system("python demo.py share")
else:
    print("❌ layerdivider のフォルダが見つかりません。")


⚙️ 公開URLの発行を強制的に有効化しています...
🛠️ 修正が完了しました！ツールを起動します...


In [27]:
import os
import shutil

print("⚙️ 既存のファイル競合を解消しています...")

# 1. 衝突している古いファイルやフォルダを完全に強制削除する
target_model_path = "/content/layerdivider/models/sam_vit_h_4b8939.pth"
if os.path.exists(target_model_path) or os.path.islink(target_model_path):
    if os.path.isdir(target_model_path) and not os.path.islink(target_model_path):
        shutil.rmtree(target_model_path)
    else:
        os.remove(target_model_path)
    print("🧹 古い競合ファイルを削除しました。")

# 2. 正しいモデル（5.11GB）へのリンクを再確立
model_cache_path = "/root/.cache/huggingface/hub/models--facebook--sam-vit-huge/snapshots/87aecf0df4ce6b30cd7de76e87673c49644bdf67/pytorch_model.bin"
!mkdir -p /content/layerdivider/models
os.symlink(model_cache_path, target_model_path)
print("📦 AIモデルの配置リンクを正常に確立しました！")

# 3. 起動
print("🚀 layerdivider を再起動します...\n" + "="*40)
os.chdir("/content/layerdivider")
os.system("python demo.py share")


⚙️ 既存のファイル競合を解消しています...
🧹 古い競合ファイルを削除しました。
📦 AIモデルの配置リンクを正常に確立しました！
🚀 layerdivider を再起動します...


2

In [40]:
import os
import shutil
from huggingface_hub import hf_hub_download

print("⚙️ 破損したモデル(2.39GB)のクリーンアップと完全なモデル(5.11GB)の再取得を開始します...")

# 1. 汚れたコードをリセット
%cd /content/layerdivider
!git checkout ldivider/ld_utils.py demo.py 2>/dev/null || git checkout ddivider/ld_utils.py demo.py

# 2. 途中で壊れて保存されたキャッシュファイルや古いショートカットを根こそぎ完全削除
!rm -rf /root/.cache/huggingface/hub/models--facebook--sam-vit-huge
target_model_path = "/content/layerdivider/models/sam_vit_h_4b8939.pth"
if os.path.exists(target_model_path) or os.path.islink(target_model_path):
    os.remove(target_model_path)

# 3. 必要なフォルダの作成
os.makedirs("/content/layerdivider/models", exist_ok=True)
os.makedirs("/content/layerdivider/output/tmp/seg_layer", exist_ok=True)

# 4. 【最新版】Meta公式リポジトリから、5.11GBの完全なファイルを100%確実にダウンロード
print("⏳ 5.11 GBの完全なモデルデータを再ダウンロード中...（2〜4分かかります）")
try:
    model_path = hf_hub_download(
        repo_id="facebook/sam-vit-huge",
        filename="pytorch_model.bin",
        force_download=True  # 壊れたキャッシュを無視して強制的に最初から落とし直す
    )

    # 5. 正しいファイル名で確実にショートカットリンクを作成
    os.symlink(model_path, target_model_path)
    current_size = os.path.getsize(target_model_path) / (1024**3)
    print(f"✅ ついに本物の完全モデルの配置に成功しました！確定サイズ: {current_size:.2f} GB")

except Exception as e:
    print(f"❌ ダウンロード中にエラーが発生しました: {e}")

# 6. Python 3.13環境における、内部のバグと読み込みの安全エラー（weights_only）を上書き修正
ld_utils_path = "/content/layerdivider/ldivider/ld_utils.py"
if os.path.exists(ld_utils_path):
    with open(ld_utils_path, "r") as f:
        code = f.read()

    fix_line = 'os.makedirs(os.path.join(output_dir, "tmp", "seg_layer"), exist_ok=True)'
    if fix_line not in code:
        code = code.replace('def init_seg(output_dir):', f'def init_seg(output_dir):\n    {fix_line}')

    code = code.replace('torch.load(model_path)', 'torch.load(model_path, weights_only=False)')
    code = code.replace('torch.load(model_path, map_location="cpu")', 'torch.load(model_path, map_location="cpu", weights_only=False)')

    with open(ld_utils_path, "w") as f:
        f.write(code)
    print("🛠️ 内部互換性パッチをシステムに適用しました。")

# 7. 画面の乗っ取りを防止しつつ、公開URLを発行する設定をdemo.pyに埋め込む
demo_file_path = "/content/layerdivider/demo.py"
with open(demo_file_path, "r") as f:
    demo_code = f.read()

demo_code = demo_code.replace("demo.launch()", "demo.launch(share=True, debug=True, inline=False)")
demo_code = demo_code.replace("launch(server_name='0.0.0.0')", "launch(server_name='0.0.0.0', share=True, debug=True, inline=False)")

with open(demo_file_path, "w") as f:
    f.write(demo_code)

# 8. 起動
print("🚀 すべての障害を取り除きました。layerdivider を再起動します...\n" + "="*40)
!python demo.py share


⚙️ 破損したモデル(2.39GB)のクリーンアップと完全なモデル(5.11GB)の再取得を開始します...
/content/layerdivider
⏳ 5.11 GBの完全なモデルデータを再ダウンロード中...（2〜4分かかります）


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.56GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

✅ ついに本物の完全モデルの配置に成功しました！確定サイズ: 2.39 GB
🛠️ 内部互換性パッチをシステムに適用しました。
🚀 すべての障害を取り除きました。layerdivider を再起動します...
* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://58176c0d4aed8a4ff7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/di

In [41]:
import os

base = "/content/layerdivider/output/tmp/seg_layer"

print("ディレクトリ存在:", os.path.isdir(base))

if os.path.isdir(base):
    print("ファイル一覧:", os.listdir(base))

ディレクトリ存在: True
ファイル一覧: ['sample.txt']


In [42]:
import os

base = "/content/layerdivider"

# マスク関連ファイルを検索
for root, dirs, files in os.walk(base):
    for name in files:
        if "mask" in name.lower() or name.endswith(".pkl"):
            print(os.path.join(root, name))

# エラー発生箇所のコードを表示
for path, start, end in [
    (f"{base}/demo.py", 25, 65),
    (f"{base}/ldivider/ld_utils.py", 55, 90),
]:
    print(f"\n===== {path} =====")
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}", end="")


===== /content/layerdivider/demo.py =====
25:         self.demo = gr.Blocks()
26:         
27:     def segment_image(self, input_image, pred_iou_thresh, stability_score_thresh, crop_n_layers, crop_n_points_downscale_factor, min_mask_region_area):
28:         mask_generator = get_mask_generator(pred_iou_thresh, stability_score_thresh, min_mask_region_area, model_dir, "demo")
29:         masks = get_masks(pil2cv(input_image), mask_generator)
30:         input_image.putalpha(255)
31:         masked_image = show_anns(input_image, masks, output_dir)
32:         return masked_image
33: 
34:     def divide_layer(self, divide_mode, input_image, loops, init_cluster, ciede_threshold, blur_size, layer_mode, h_split, v_split, n_cluster, alpha, th_rate, split_bg, area_th):
35:         if divide_mode == "segment_mode":
36:             return self.segment_divide(input_image, loops, init_cluster, ciede_threshold, blur_size, layer_mode, h_split, v_split, n_cluster, alpha, th_rate, split_bg, area_th)
3

In [46]:
from pathlib import Path

path = Path("/content/layerdivider/ldivider/ld_segment.py")
source = path.read_text(encoding="utf-8")

old = '''def show_anns(image, masks, output_dir):
    if len(masks) == 0:
        return
    sorted_masks = sorted(masks, key=(lambda x: x['area']), reverse=True)
    # pickle化してファイルに書き込み
    with open(f'{output_dir}/tmp/seg_layer/sorted_masks.pkl', 'wb') as f:
        pickle.dump(sorted_masks, f)'''

new = '''def show_anns(image, masks, output_dir):
    import os

    if len(masks) == 0:
        raise ValueError(
            "SAMがマスクを1件も生成しませんでした。"
            "入力画像とSAMの設定を確認してください。"
        )

    sorted_masks = sorted(masks, key=lambda x: x['area'], reverse=True)

    save_dir = os.path.join(output_dir, "tmp", "seg_layer")
    os.makedirs(save_dir, exist_ok=True)
    pkl_path = os.path.join(save_dir, "sorted_masks.pkl")

    with open(pkl_path, "wb") as f:
        pickle.dump(sorted_masks, f)'''

if old not in source:
    print("置換対象が見つかりません。ファイルは変更していません。")
else:
    backup = path.with_suffix(".py.bak")
    backup.write_text(source, encoding="utf-8")
    path.write_text(source.replace(old, new, 1), encoding="utf-8")
    print("修正しました。バックアップ:", backup)

修正しました。バックアップ: /content/layerdivider/ldivider/ld_segment.py.bak


In [50]:
p = "/content/layerdivider/demo.py"

with open(p, encoding="utf-8") as f:
    lines = f.readlines()

for i in range(100, min(180, len(lines))):
    print(f"{i+1}: {lines[i]}", end="")

101:                 output_dir,
102:                 layer_mode,
103:             )
104:             base_layer_list = [cv2pil(layer) for layer in base_layer_list]
105:             return [image, base_image], base_layer_list, bright_layer_list, shadow_layer_list, filename
106:         elif layer_mode == "normal":
107:             base_layer_list, bright_layer_list, shadow_layer_list = get_normal_layer(self.input_image, df)
108:             filename = save_psd(
109:                 self.input_image,
110:                 [base_layer_list, bright_layer_list, shadow_layer_list],
111:                 ["base", "bright", "shadow"],
112:                 [BlendMode.normal, BlendMode.normal, BlendMode.normal],
113:                 output_dir,
114:                 layer_mode,
115:             )
116:             return [image, base_image], base_layer_list, bright_layer_list, shadow_layer_list, filename
117:         else:
118:             return None
119: 
120:     def launch(self, share):
121:   

In [ ]:
!python demo.py share

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://c02d675151f62ea119.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
71it [00:10,  6.52it/s]
100% 24/24 [01:11<00:00,  3.00s/it]
100% 24/24 [01:27<00:00,  3.64s/it]
100% 24/24 [01:23<00:00,  3.49s/it]
